In [1]:
import pandas as pd

df = pd.read_csv("Source Files/raw_delivery.csv")

In [2]:
df.head()


,Delivery_ID,Sales_ID,Promised_Date,Delivered_Date,Delay_Days
0,D0001,S0086,2026-04-03,2026-04-07,4
1,D0002,S0282,2026-07-25,2026-07-26,1
2,D0003,S0259,2026-06-22,2026-06-27,5
3,D0004,S0337,2026-03-22,2026-03-30,8
4,D0005,S0274,2026-08-15,2026-08-16,1


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Delivery_ID     350 non-null    str  
 1   Sales_ID        350 non-null    str  
 2   Promised_Date   350 non-null    str  
 3   Delivered_Date  350 non-null    str  
 4   Delay_Days      350 non-null    int64
dtypes: int64(1), str(4)
memory usage: 24.1 KB


In [4]:
df.describe()

,Delay_Days
count,350.000000
mean,2.811429
std,3.193634
min,0.000000
25%,0.000000
50%,2.000000
75%,5.000000
max,20.000000


In [5]:
# Convert date columns into datetime format - str to datetime
df["Promised_Date"] = pd.to_datetime(df["Promised_Date"])
df["Delivered_Date"] = pd.to_datetime(df["Delivered_Date"])

# Check data types and missing values
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())

# Check duplicate delivery IDs
print("\nDuplicate Delivery IDs:", df["Delivery_ID"].duplicated().sum())

# Check delay distribution
print("\nDelay summary:")
print(df["Delay_Days"].value_counts().sort_index())

Delivery_ID                  str
Sales_ID                     str
Promised_Date     datetime64[us]
Delivered_Date    datetime64[us]
Delay_Days                 int64
dtype: object

Missing values:
Delivery_ID       0
Sales_ID          0
Promised_Date     0
Delivered_Date    0
Delay_Days        0
dtype: int64

Duplicate Delivery IDs: 0

Delay summary:
Delay_Days
0     123
1      32
2      34
3      42
4      29
5      34
6      17
7      10
8      11
9       4
10      4
11      4
12      1
13      1
15      2
17      1
20      1
Name: count, dtype: int64


In [6]:
# Categorize delivery performance
df["Delivery_Status"] = df["Delay_Days"].apply(
    lambda x: "Delayed" if x > 0 else "On Time"
)

# Count deliveries by status
print(df["Delivery_Status"].value_counts())

# Overall delivery KPIs
total_deliveries = len(df)
delayed_deliveries = (df["Delay_Days"] > 0).sum()

print("Total deliveries:", total_deliveries)
print("Delayed deliveries:", delayed_deliveries)
print("On-time deliveries:", total_deliveries - delayed_deliveries)
print("Delay rate (%):", round(delayed_deliveries / total_deliveries * 100, 2))
print("Average delay (days):", round(df["Delay_Days"].mean(), 2))

Delivery_Status
Delayed    227
On Time    123
Name: count, dtype: int64
Total deliveries: 350
Delayed deliveries: 227
On-time deliveries: 123
Delay rate (%): 64.86
Average delay (days): 2.81


In [7]:

# Group deliveries by promised month
df["Promised_Month"] = df["Promised_Date"].dt.to_period("M")

monthly_delivery = df.groupby("Promised_Month").agg(
    Total_Deliveries=("Delivery_ID", "nunique"),
    Delayed_Deliveries=("Delay_Days", lambda x: (x > 0).sum()),
    Average_Delay_Days=("Delay_Days", "mean")
)

# Calculate delay percentage
monthly_delivery["Delay_Rate_%"] = (
    monthly_delivery["Delayed_Deliveries"]
    / monthly_delivery["Total_Deliveries"] * 100
).round(2)

# Display worst-performing months first
monthly_delivery = monthly_delivery.sort_values(
    by="Delay_Rate_%", ascending=False
)

monthly_delivery.round(2)


,Total_Deliveries,Delayed_Deliveries,Average_Delay_Days,Delay_Rate_%
Promised_Month,,,,
2026-06,52,39,2.96,75.00
2026-08,30,22,2.47,73.33
2026-09,21,15,3.33,71.43
2026-05,33,23,2.88,69.70
2026-02,27,17,3.30,62.96
2026-07,50,31,2.72,62.00
2026-01,47,29,2.68,61.70
2026-03,49,29,2.63,59.18
2026-04,41,22,2.71,53.66
